# FaceFusion Colab (T4)

Runtime: **GPU T4**. Bấm **Setup** một lần, rồi **Run**. Headless swap luôn, không mở UI.

T4: CUDA only. Temp: **MyDrive/facefusion/temp**. Output: **MyDrive/output** (tên hash tự tạo). `memory`: không extract jpeg, ghi thẳng mp4.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

print('T4 Setup start', flush=True)
sys.stdout.flush()

from google.colab import drive

mydrive = Path('/content/drive/MyDrive')
if mydrive.exists():
    print('Drive already mounted', mydrive, flush=True)
else:
    print('Mount Drive — bấm Connect ngay khi popup hiện', flush=True)
    try:
        drive.mount('/content/drive')
    except ValueError:
        print('mount lần 1 fail, retry force_remount', flush=True)
        drive.mount('/content/drive', force_remount=True)
    if not mydrive.exists():
        raise SystemExit('Drive mount failed. Runtime → Disconnect and delete runtime, chạy lại Setup, bấm Connect ngay.')
print('Drive OK', mydrive, flush=True)


def run(cmd, check=True, **kwargs):
    print('+', ' '.join(str(part) for part in cmd), flush=True)
    result = subprocess.run(cmd, **kwargs)
    if check and result.returncode != 0:
        raise SystemExit(f'Lệnh thất bại ({result.returncode}): {" ".join(str(part) for part in cmd)}')
    return result


def ort_providers():
    probe = subprocess.run(
        [sys.executable, '-c', 'import onnxruntime as ort; print(",".join(ort.get_available_providers()))'],
        capture_output=True, text=True
    )
    if probe.returncode != 0:
        err = (probe.stderr or '') + (probe.stdout or '')
        if 'No module named' in err:
            print('onnxruntime chưa cài — sẽ chạy install.py', flush=True)
        else:
            print(probe.stdout, probe.stderr, flush=True)
        return set()
    return set(p for p in probe.stdout.strip().split(',') if p)


def link_dir(local, target):
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        if os.path.realpath(local) != str(target):
            local.unlink()
    elif local.exists():
        for item in local.iterdir():
            dest = target / item.name
            if not dest.exists():
                shutil.move(str(item), str(dest))
        shutil.rmtree(local)
    if not local.exists():
        os.symlink(target, local)
    print(local.name, '->', os.path.realpath(local), flush=True)


gpu = run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False)
if gpu.returncode != 0:
    raise SystemExit('Không thấy GPU. Runtime → Change runtime type → GPU T4.')

print('apt-get update (im một lúc là bình thường)', flush=True)
run(['apt-get', 'update', '-qq'])
print('apt-get install ffmpeg curl', flush=True)
run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'curl'])

DRIVE = Path('/content/drive/MyDrive')
SOURCE = DRIVE / 'input' / 'face' / 'thao.JPG'
TARGET = DRIVE / 'input' / 'phim'
OUTPUT_DIR = DRIVE / 'output'
TEMP = Path('/content/facefusion/temp')
JOBS = DRIVE / 'facefusion' / 'jobs'
ASSETS = DRIVE / 'facefusion' / 'assets'
CACHES = DRIVE / 'facefusion' / 'caches'
REPO = Path('/content/facefusionnnnnn')
CLONE_URL = 'https://github.com/hunguet123/facefusionnnnnn.git'

if not SOURCE.exists():
    alt = SOURCE.with_suffix('.jpg')
    if alt.exists():
        SOURCE = alt

for folder in [OUTPUT_DIR, TEMP, JOBS, ASSETS, ASSETS / 'models', CACHES]:
    folder.mkdir(parents=True, exist_ok=True)

assert SOURCE.exists(), f'Thiếu source: {SOURCE}'
TARGET.mkdir(parents=True, exist_ok=True)

os.environ['FACEFUSION_KEEP_TEMP'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

if (REPO / '.git').exists():
    run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'])
    run(['git', '-C', str(REPO), 'reset', '--hard', 'FETCH_HEAD'])
else:
    run(['git', 'clone', '--branch', 'main', '--depth', '1', CLONE_URL, str(REPO)])
os.chdir(REPO)

link_dir(REPO / '.assets', ASSETS)
link_dir(REPO / '.caches', CACHES)

if 'CUDAExecutionProvider' in ort_providers():
    print('CUDA already installed, skip install.py', flush=True)
else:
    run([sys.executable, 'install.py', 'cuda@12', '--skip-conda'])
    if 'CUDAExecutionProvider' not in ort_providers():
        print('CUDA EP chưa có sau cuda@12, thử cuda@13', flush=True)
        run([sys.executable, 'install.py', 'cuda@13', '--skip-conda'])
    if 'CUDAExecutionProvider' not in ort_providers():
        raise SystemExit('onnxruntime-gpu không có CUDAExecutionProvider. Setup thất bại.')

providers = ort_providers()
print('ORT providers:', sorted(providers), flush=True)
print('TensorRT: SKIP — T4 dùng CUDA only', flush=True)

core_path = REPO / 'facefusion' / 'workflows' / 'core.py'
video_path = REPO / 'facefusion' / 'workflows' / 'to_video.py'
core_text = core_path.read_text()
if 'FACEFUSION_KEEP_TEMP' not in core_text:
    if not core_text.startswith('import os'):
        core_text = 'import os\n' + core_text
    old = (
        'def clear() -> ErrorCode:\n'
        "\tif clear_temp_directory(state_manager.get_item('target_path')):\n"
        "\t\tlogger.debug(translator.get('clearing_temp'), __name__)\n"
        '\treturn 0'
    )
    new = (
        'def clear() -> ErrorCode:\n'
        "\tif os.getenv('FACEFUSION_KEEP_TEMP') == '1':\n"
        "\t\tlogger.info('keeping temp on drive', __name__)\n"
        '\t\treturn 0\n'
        "\tif clear_temp_directory(state_manager.get_item('target_path')):\n"
        "\t\tlogger.debug(translator.get('clearing_temp'), __name__)\n"
        '\treturn 0'
    )
    if old not in core_text:
        raise SystemExit('Không patch được core.clear — source đã đổi.')
    core_path.write_text(core_text.replace(old, new, 1))
    print('patched core.clear', flush=True)
else:
    print('core.clear already patched', flush=True)

video_text = video_path.read_text()
if 'FACEFUSION_KEEP_TEMP' not in video_text:
    if 'import os\n' not in video_text.split('def extract_frames', 1)[0]:
        video_text = video_text.replace('from collections import deque\n', 'import os\nfrom collections import deque\n', 1)
    old = (
        'def extract_frames() -> ErrorCode:\n'
        "\ttrim_frame_start, trim_frame_end = restrict_trim_frame(state_manager.get_item('target_path'), state_manager.get_item('trim_frame_start'), state_manager.get_item('trim_frame_end'))\n"
    )
    new = (
        'def extract_frames() -> ErrorCode:\n'
        "\texisting_frames = resolve_temp_frame_set(state_manager.get_item('target_path'))\n"
        "\tif os.getenv('FACEFUSION_KEEP_TEMP') == '1' and existing_frames:\n"
        "\t\tlogger.info('resume: skip extract, frames already on drive (' + str(len(existing_frames)) + ')', __name__)\n"
        '\t\treturn 0\n'
        "\ttrim_frame_start, trim_frame_end = restrict_trim_frame(state_manager.get_item('target_path'), state_manager.get_item('trim_frame_start'), state_manager.get_item('trim_frame_end'))\n"
    )
    if old not in video_text:
        raise SystemExit('Không patch được extract_frames — source đã đổi.')
    video_path.write_text(video_text.replace(old, new, 1))
    print('patched to_video.extract_frames', flush=True)
else:
    print('extract_frames already patched', flush=True)

exit_path = REPO / 'facefusion' / 'exit_helper.py'
exit_text = exit_path.read_text()
if 'FACEFUSION_KEEP_TEMP' not in exit_text:
    old = (
        "\tif state_manager.get_item('target_path'):\n"
        "\t\tclear_temp_directory(state_manager.get_item('target_path'))\n"
    )
    new = (
        "\tif state_manager.get_item('target_path') and os.getenv('FACEFUSION_KEEP_TEMP') != '1':\n"
        "\t\tclear_temp_directory(state_manager.get_item('target_path'))\n"
    )
    if old not in exit_text:
        raise SystemExit('Không patch được exit_helper')
    exit_path.write_text(exit_text.replace(old, new, 1))
    print('patched exit_helper', flush=True)
else:
    print('exit_helper already patched', flush=True)



def patch_once(path, marker, old, new, also=None):
    text = path.read_text()
    if marker in text:
        print(path.name, 'already patched', flush=True)
        return
    if also:
        text = text.replace(also[0], also[1], 1)
    if old not in text:
        raise SystemExit('Không patch được ' + str(path))
    path.write_text(text.replace(old, new, 1))
    print('patched', path.name, flush=True)

video_wf = REPO / 'facefusion' / 'workflows' / 'image_to_video.py'
patch_once(
    video_wf,
    "step: ' + task_name",
    "	process_manager.start()\n\n	for task in tasks:\n		error_code = task() #type:ignore[operator]\n",
    "	process_manager.start()\n\n	for task in tasks:\n		task_name = getattr(getattr(task, 'func', task), '__name__', str(task))\n		logger.info('step: ' + task_name, __name__)\n		error_code = task() #type:ignore[operator]\n",
    also=("from facefusion import process_manager, state_manager\n", "from facefusion import logger, process_manager, state_manager\n"),
)
core_py = REPO / 'facefusion' / 'core.py'
patch_once(
    core_py,
    'loading processor ',
    "def processors_pre_check() -> bool:\n	for processor_module in get_processors_modules(state_manager.get_item('processors')):\n		if not processor_module.pre_check():\n			return False\n	return True\n",
    "def processors_pre_check() -> bool:\n	for processor_module in get_processors_modules(state_manager.get_item('processors')):\n		logger.info('loading processor ' + processor_module.__name__, __name__)\n		if not processor_module.pre_check():\n			return False\n	return True\n",
)
patch_once(
    core_py,
    'pre_process ',
    "		for processor_module in get_processors_modules(state_manager.get_item('processors')):\n			if not processor_module.pre_process('output'):\n				return 2\n",
    "		for processor_module in get_processors_modules(state_manager.get_item('processors')):\n			logger.info('pre_process ' + processor_module.__name__, __name__)\n			if not processor_module.pre_process('output'):\n				return 2\n",
)

print('SOURCE', SOURCE, flush=True)
print('TARGET', TARGET, flush=True)
print('OUTPUT', OUTPUT_DIR, flush=True)
print('TEMP  ', TEMP, flush=True)
print('Setup xong. Chạy cell Run để swap headless.', flush=True)


In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/facefusionnnnnn')
if not (REPO / '.git').exists():
    raise SystemExit('Chưa có repo. Chạy cell Setup trước.')

def run(cmd):
    print('+', ' '.join(cmd), flush=True)
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise SystemExit(result.returncode)
    return result

run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'])
run(['git', '-C', str(REPO), 'reset', '--hard', 'FETCH_HEAD'])
run(['git', '-C', str(REPO), 'log', '-1', '--oneline'])
print('Pull xong. Chạy lại cell UI hoặc Run — không cần Setup.', flush=True)


In [ ]:
import hashlib
import os
import re
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

# === chỉ đổi tên video ở đây ===
TARGET_NAME = 'phim2_000.mp4'
SOURCE_NAME = 'thao.JPG'
# ==============================

print('T4 headless start', flush=True)
mydrive = Path('/content/drive/MyDrive')
if mydrive.exists():
    print('Drive already mounted', mydrive, flush=True)
else:
    print('Mount Drive — bấm Connect ngay khi popup hiện', flush=True)
    try:
        drive.mount('/content/drive')
    except ValueError:
        drive.mount('/content/drive', force_remount=True)
    if not mydrive.exists():
        raise SystemExit('Drive mount failed. Runtime → Disconnect and delete runtime, chạy lại, bấm Connect ngay.')

REPO = Path('/content/facefusionnnnnn')
if not (REPO / 'facefusion.py').exists():
    raise SystemExit('Chưa có repo. Chạy cell Setup trước.')

os.chdir(REPO)
os.environ['FACEFUSION_KEEP_TEMP'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['GRADIO_ANALYTICS_ENABLED'] = '0'

gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU', (gpu_probe.stdout or '').strip(), flush=True)

DRIVE = Path('/content/drive/MyDrive')
SOURCE = DRIVE / 'input' / 'face' / SOURCE_NAME
if not SOURCE.exists():
    SOURCE = SOURCE.with_suffix('.jpg')
target_candidates = [
    DRIVE / 'input' / 'phim' / TARGET_NAME,
    DRIVE / 'input' / TARGET_NAME,
    DRIVE / 'phim' / TARGET_NAME,
]
TARGET = next((p for p in target_candidates if p.exists()), target_candidates[0])
OUTPUT_DIR = DRIVE / 'output'
TEMP = Path('/content/facefusion/temp')
JOBS = DRIVE / 'facefusion' / 'jobs'

assert SOURCE.exists(), f'Thiếu source: {SOURCE}'
assert TARGET.exists(), f'Thiếu target: {TARGET}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP.mkdir(parents=True, exist_ok=True)
JOBS.mkdir(parents=True, exist_ok=True)
print('SOURCE', SOURCE, flush=True)
print('TARGET', TARGET, flush=True)
OUTPUT = OUTPUT_DIR / TARGET.name
print('OUTPUT folder', OUTPUT_DIR, flush=True)
print('OUTPUT file  ', OUTPUT, flush=True)
print('TEMP  ', TEMP, flush=True)


def patch_once(path, marker, old, new, also=None):
    text = path.read_text()
    if marker in text:
        print(path.name, 'already patched', flush=True)
        return
    if also:
        text = text.replace(also[0], also[1], 1)
    if old not in text:
        raise SystemExit('Không patch được ' + str(path))
    path.write_text(text.replace(old, new, 1))
    print('patched', path.name, flush=True)


core_wf = REPO / 'facefusion' / 'workflows' / 'core.py'
core_text = core_wf.read_text()
if 'FACEFUSION_KEEP_TEMP' not in core_text:
    if not core_text.startswith('import os'):
        core_text = 'import os\n' + core_text
    old = (
        'def clear() -> ErrorCode:\n'
        "\tif clear_temp_directory(state_manager.get_item('target_path')):\n"
        "\t\tlogger.debug(translator.get('clearing_temp'), __name__)\n"
        '\treturn 0'
    )
    new = (
        'def clear() -> ErrorCode:\n'
        "\tif os.getenv('FACEFUSION_KEEP_TEMP') == '1':\n"
        "\t\tlogger.info('keeping temp on drive', __name__)\n"
        '\t\treturn 0\n'
        "\tif clear_temp_directory(state_manager.get_item('target_path')):\n"
        "\t\tlogger.debug(translator.get('clearing_temp'), __name__)\n"
        '\treturn 0'
    )
    if old not in core_text:
        raise SystemExit('Không patch được core.clear')
    core_wf.write_text(core_text.replace(old, new, 1))

video_path = REPO / 'facefusion' / 'workflows' / 'to_video.py'
video_text = video_path.read_text()
if 'FACEFUSION_KEEP_TEMP' not in video_text:
    if 'import os\n' not in video_text.split('def extract_frames', 1)[0]:
        video_text = video_text.replace('from collections import deque\n', 'import os\nfrom collections import deque\n', 1)
    old = (
        'def extract_frames() -> ErrorCode:\n'
        "\ttrim_frame_start, trim_frame_end = restrict_trim_frame(state_manager.get_item('target_path'), state_manager.get_item('trim_frame_start'), state_manager.get_item('trim_frame_end'))\n"
    )
    new = (
        'def extract_frames() -> ErrorCode:\n'
        "\texisting_frames = resolve_temp_frame_set(state_manager.get_item('target_path'))\n"
        "\tif os.getenv('FACEFUSION_KEEP_TEMP') == '1' and existing_frames:\n"
        "\t\tlogger.info('resume: skip extract, frames already on drive (' + str(len(existing_frames)) + ')', __name__)\n"
        '\t\treturn 0\n'
        "\ttrim_frame_start, trim_frame_end = restrict_trim_frame(state_manager.get_item('target_path'), state_manager.get_item('trim_frame_start'), state_manager.get_item('trim_frame_end'))\n"
    )
    if old not in video_text:
        raise SystemExit('Không patch được extract_frames')
    video_path.write_text(video_text.replace(old, new, 1))

exit_path = REPO / 'facefusion' / 'exit_helper.py'
exit_text = exit_path.read_text()
if 'FACEFUSION_KEEP_TEMP' not in exit_text:
    old = (
        "\tif state_manager.get_item('target_path'):\n"
        "\t\tclear_temp_directory(state_manager.get_item('target_path'))\n"
    )
    new = (
        "\tif state_manager.get_item('target_path') and os.getenv('FACEFUSION_KEEP_TEMP') != '1':\n"
        "\t\tclear_temp_directory(state_manager.get_item('target_path'))\n"
    )
    if old not in exit_text:
        raise SystemExit('Không patch được exit_helper')
    exit_path.write_text(exit_text.replace(old, new, 1))
    print('patched exit_helper', flush=True)


probe = subprocess.run(
    [sys.executable, '-c', 'import onnxruntime as ort; print(",".join(ort.get_available_providers()))'],
    capture_output=True, text=True
)
if probe.returncode != 0 or 'CUDAExecutionProvider' not in probe.stdout:
    print(probe.stdout, probe.stderr, flush=True)
    raise SystemExit('Không có CUDA. Chạy lại cell Setup trên runtime GPU T4.')
print('ORT providers:', sorted(p for p in probe.stdout.strip().split(',') if p), flush=True)

execution_providers = ['cuda']
print('TensorRT: SKIP — T4 CUDA only', flush=True)

video_encoder = 'libx264'
ini_text = Path('facefusion.ini').read_text()
ini_text = re.sub(r'^execution_providers\s*=.*', 'execution_providers = ' + ' '.join(execution_providers), ini_text, flags=re.M)
ini_text = re.sub(r'^output_video_encoder\s*=.*', 'output_video_encoder = ' + video_encoder, ini_text, flags=re.M)
ini_text = re.sub(r'^download_providers\s*=.*', 'download_providers = huggingface github', ini_text, flags=re.M)
ini_text = re.sub(r'^open_browser\s*=.*', 'open_browser = false', ini_text, flags=re.M)
ini_text = re.sub(r'^execution_thread_count\s*=.*', 'execution_thread_count = 10', ini_text, flags=re.M)
ini_text = re.sub(r'^workflow_strategy\s*=.*', 'workflow_strategy = memory', ini_text, flags=re.M)
Path('facefusion.colab.ini').write_text(ini_text)


def stream_live(cmd):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['PYTHONIOENCODING'] = 'utf-8'
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
        bufsize=0,
    )
    last_cr = 0.0
    carry = ''
    while True:
        chunk = process.stdout.read(256)
        if not chunk:
            break
        carry += chunk.decode('utf-8', errors='replace').replace('\r\n', '\n')
        while True:
            nl = carry.find('\n')
            cr = carry.find('\r')
            if nl == -1 and cr == -1:
                break
            if cr != -1 and (nl == -1 or cr < nl):
                line, carry = carry[:cr], carry[cr + 1:]
                now = time.time()
                if line.strip() and now - last_cr >= 1:
                    print(line, flush=True)
                    last_cr = now
            else:
                line, carry = carry[:nl], carry[nl + 1:]
                print(line, flush=True)
    if carry.strip():
        print(carry, flush=True)
    return process.wait()


cmd = [
    sys.executable, '-u', 'facefusion.py', 'headless-run',
    '--config-path', 'facefusion.colab.ini',
    '--temp-path', str(TEMP),
    '--jobs-path', str(JOBS),
    '--source-paths', str(SOURCE),
    '--target-path', str(TARGET),
    '--output-path', str(OUTPUT),
    '--execution-providers', *execution_providers,
    '--output-video-encoder', video_encoder,
    '--download-providers', 'huggingface', 'github',
    '--execution-thread-count', '10',
    '--workflow-strategy', 'memory',
    '--log-level', 'debug',
]
print('=== HEADLESS SWAP — log bên dưới ===', flush=True)
print(' '.join(cmd), flush=True)
code = stream_live(cmd)
print('exit code', code, flush=True)
print('Output folder:', OUTPUT_DIR, flush=True)
print('Output file  :', OUTPUT, 'exists=', OUTPUT.exists(), flush=True)
if code != 0:
    raise SystemExit(code)
